# SemEval-2026 Task 13A — AI Code Detection Pipeline
> **Tác giả**: 25C11066  
> Mỗi cell gọi một module Python riêng biệt. Chỉ cần chỉnh `config.py`.

## Flow
```
0. Setup & Config
1. OOD Diagnosis  (diag_shift.py)
2. Feature Extraction  (feature_extractor.py)
3. Train GBM Ensemble  (train_gbm.py)
4. Fine-tune CodeBERT  (train_codebert.py)
5. Train IF+CNB  (train_ifcnb.py)
6. Soft-Voting Ensemble  (ensemble.py)
```

In [ ]:
# ── Cài đặt (chỉ cần chạy lần đầu) ─────────────────────────────────
# !pip install -r requirements.txt -q

import os, sys, logging
import numpy as np
import pandas as pd

# Thêm thư mục semeval/ vào path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s')
print('✓ Imports OK')

In [ ]:
# ── Config — Chỉnh tất cả tham số ở ĐÂY ────────────────────────────
from config import CFG

cfg = CFG()

# Override nếu chạy trên Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
if IS_KAGGLE:
    cfg.train_data  = '/kaggle/input/semeval2026/train.parquet'
    cfg.val_data    = '/kaggle/input/semeval2026/validation.parquet'
    cfg.test_data   = '/kaggle/input/semeval2026/test.parquet'
    cfg.output_dir  = '/kaggle/working/outputs'
    cfg.max_train   = 100_000
    cfg.epochs      = 3

os.makedirs(cfg.output_dir, exist_ok=True)
print(f'Model: {cfg.model_name}')
print(f'Output: {cfg.output_dir}')
print(f'FP16: {cfg.fp16}')

## 1. OOD Feature Shift Diagnosis
Tính Cohen's |d| cho từng feature giữa train và test.
Features có **|d| > 1.0** là ngôn ngữ-proxy — cần loại bỏ.


In [ ]:
from diag_shift import compute_shift_report, plot_cohens_d

# ⚠️  Cần có train_features và test_features đã extract sẵn
TRAIN_FEAT = cfg.train_data.replace('.parquet', '_features.parquet')
TEST_FEAT  = cfg.test_data.replace('.parquet', '_features.parquet') if cfg.test_data else None

if TEST_FEAT and os.path.exists(TRAIN_FEAT) and os.path.exists(TEST_FEAT):
    shift_df = compute_shift_report(
        TRAIN_FEAT, TEST_FEAT,
        top_n=25,
        save_csv=os.path.join(cfg.output_dir, 'shift_report.csv')
    )
    # Vẽ biểu đồ
    plot_cohens_d(shift_df, top_n=30,
                  save_path=os.path.join(cfg.output_dir, 'cohens_d.png'))
    display(shift_df.head(20))
else:
    print('Bỏ qua: chưa có feature files. Chạy feature extraction trước.')

## 2. Feature Extraction (75+ handcrafted features)
Trích xuất song song trên train / val / test.


In [ ]:
from feature_extractor import extract_all_features
from data_utils import load_dataframe

# Train features
train_df = load_dataframe(cfg.train_data, cfg.max_train, 'Train', cfg.seed)
print('Đang trích xuất train features...')
train_feat = extract_all_features(train_df['code'], show_progress=True)
train_feat['label'] = train_df['label'].values
train_feat.to_parquet(TRAIN_FEAT, index=False)
print(f'Train features: {train_feat.shape} → {TRAIN_FEAT}')

# Val features
VAL_FEAT = cfg.val_data.replace('.parquet', '_features.parquet')
val_df = load_dataframe(cfg.val_data, cfg.max_val, 'Val', cfg.seed)
val_feat = extract_all_features(val_df['code'], show_progress=True)
val_feat['label'] = val_df['label'].values
val_feat.to_parquet(VAL_FEAT, index=False)
print(f'Val features: {val_feat.shape} → {VAL_FEAT}')

## 3. Train Language-Robust GBM Ensemble
XGBoost + LightGBM + CatBoost với 75 features (loại 7 OOD features).


In [ ]:
from train_gbm import run_gbm

gbm_tau, val_proba_gbm = run_gbm(
    train_feat_path = TRAIN_FEAT,
    val_feat_path   = VAL_FEAT,
    cfg             = cfg,
    test_feat_path  = TEST_FEAT,
)
print(f'GBM best τ = {gbm_tau:.3f}')
print(f'val_proba_gbm: mean={val_proba_gbm.mean():.3f}, std={val_proba_gbm.std():.3f}')

## 4. Fine-tune CodeBERT
microsoft/codebert-base với Trainer API. Early stopping trên Macro F1.


In [ ]:
from train_codebert import run_codebert

codebert_tau, val_proba_codebert = run_codebert(cfg)
print(f'CodeBERT best τ = {codebert_tau:.3f}')
print(f'val_proba_codebert: mean={val_proba_codebert.mean():.3f}')

## 5. Train IsolationForest + ComplementNB
20 style-only features — ít bị OOD shift nhất.
IF phát hiện code 'quá hoàn hảo' như outlier (ngôn ngữ-bất biến).


In [ ]:
from train_ifcnb import run_ifcnb
from data_utils import load_dataframe

train_df_full = load_dataframe(cfg.train_data, cfg.max_train, 'Train', cfg.seed)
val_df_full   = load_dataframe(cfg.val_data,   cfg.max_val,   'Val',   cfg.seed)
test_df_full  = pd.read_parquet(cfg.test_data) if cfg.test_data else None

val_proba_ifcnb = run_ifcnb(train_df_full, val_df_full, cfg, test_df=test_df_full)
print(f'val_proba_ifcnb: mean={val_proba_ifcnb.mean():.3f}')

## 6. Soft-Voting Ensemble
Rank-normalize → weighted average → quantile threshold calibration.


In [ ]:
from ensemble import run_ensemble
from data_utils import load_dataframe

# Load val labels (full)
val_full = load_dataframe(cfg.val_data, None, 'Full-Val', cfg.seed)
y_val    = val_full['label'].values

# Load test proba files
def load_npy(path):
    return np.load(path) if os.path.exists(path) else None

test_gbm      = load_npy(os.path.join(cfg.output_dir, 'test_proba_gbm.npy'))
test_codebert = load_npy(os.path.join(cfg.output_dir, 'test_proba_codebert.npy'))
test_ifcnb    = load_npy(os.path.join(cfg.output_dir, 'test_proba_ifcnb.npy'))

# Load test IDs
test_ids = None
if cfg.test_data and os.path.exists(cfg.test_data):
    test_df_tmp = pd.read_parquet(cfg.test_data)
    test_ids = test_df_tmp['ID'].tolist() if 'ID' in test_df_tmp.columns else list(range(len(test_df_tmp)))

val_blend = run_ensemble(
    val_proba_gbm      = val_proba_gbm,
    val_proba_codebert = val_proba_codebert,
    val_proba_ifcnb    = val_proba_ifcnb,
    y_val              = y_val,
    test_proba_gbm     = test_gbm,
    test_proba_codebert= test_codebert,
    test_proba_ifcnb   = test_ifcnb,
    test_ids           = test_ids,
    output_dir         = cfg.output_dir,
    submission_out     = cfg.submission_out,
)
print(f'\n✓ Submission → {cfg.submission_out}')

## 7. Tóm tắt kết quả


In [ ]:
from sklearn.metrics import f1_score
from metrics import optimize_threshold

results = {
    'GBM':      val_proba_gbm,
    'CodeBERT': val_proba_codebert,
    'IF+CNB':   val_proba_ifcnb,
    'Ensemble': val_blend,
}

print('\n' + '='*55)
print(f'{"Model":<15} {"Best τ":>8} {"Val Macro F1":>14}')
print('-'*55)
for name, proba in results.items():
    tau, f1 = optimize_threshold(proba, y_val)
    print(f'{name:<15} {tau:>8.3f} {f1:>14.4f}')
print('='*55)